In [27]:
import os
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages:Annotated[list,add_messages]
    
graph_builder=StateGraph(State)


In [28]:
graph_builder

In [29]:
from langchain_tavily import TavilySearch

tool=TavilySearch(
    max_results=5,
    tavily_api_key=os.getenv("TAVILY_API"))
tool.invoke("Provide some websites which provide VIN decoding using VIN number freely.")

{'query': 'Provide some websites which provide VIN decoding using VIN number freely.',
 'response_time': 0.9,
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.decodethevin.com/',
   'title': 'Free VIN Lookup - Vehicle VIN Decoder & Info ...',
   'content': 'Free VIN number lookup & Decoder. Learn what all the different characters in your vehicle identification number (VIN) mean with our simple guide.',
   'score': 0.99966204,
   'raw_content': None},
  {'url': 'https://www.jdpower.com/cars/vin-lookup-and-decoder',
   'title': 'VIN Lookup & Decoder',
   'content': "Enter a vehicle identification number (VIN) and unlock key details about your vehicle for free. ## How To Find Your Vehicle's VIN Number. When available, the Free VIN Lookup will include the vehicle’s year, make, model, trim, pricing, horsepower, fuel type, transmission, drivetrain, etc. ## What is a VIN Number (Vehicle Identification Number)? A VIN is a standardized 17-character

In [30]:
state = {
    # ---- EXISTING (DO NOT CHANGE) ----
    "user_input": "I want to get the vin decoding of the vin number 1HGCM82633A004352 at free of cost",
    "plan": [],
    "step_index": 0,
    "last_error": None,
    "output_content": [],
    "messages": [],

    # ---- NEW (MULTI-WEBSITE EXECUTION) ----
    "urls": [],                    # Websites discovered by Orchestrator
    "site_names": [],              # Optional (for reporting)
    "current_site_index": 0,       # Which website is active
    "retry_count": 0,              # Retries for current website
    "max_retries": 2,              # Retry limit per website
    "aggregated_results": [],      # Final results from all websites

    # ---- OPTIONAL (ROBUSTNESS) ----
    "execution_messages": [],      # Execution-only memory
    "current_model_index": 0,      # LLM rotation
    "done": False                  # Completion flag
}


In [23]:
import json


from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

llm = ChatOpenAI(
    model="gpt-oss-120b",
    api_key=os.getenv("SAMBANOVA_API_KEY"),
    base_url="https://api.sambanova.ai/v1",
    temperature=0
)

def call_llm_for_plan(system_prompt: str) -> dict:
    """
    Calls the LLM to generate a structured plan.
    This is the ONLY place where the Orchestrator talks to the LLM.
    """

    response = llm.invoke(system_prompt)
    raw_output = response.content.strip()

   
    if "```" in raw_output:
        raw_output = raw_output.split("```")[1]

    try:
        return json.loads(raw_output)
    except json.JSONDecodeError:
        raise ValueError(f"Invalid JSON returned by planner:\n{raw_output}")




In [ ]:
from typing import Dict, List


def orchestrator_agent(state: Dict) -> Dict:
    """
    Orchestrator Agent (Planner / Brain)

    Responsibilities:
    - Decide planning mode
    - Generate steps
    - Assign agents
    - Maintain state
    """
    
    

    user_input = state["user_input"].replace("{", "{{").replace("}", "}}")
    plan = state.get("plan", [])
    step_index = state.get("step_index", 0)
    last_error = state.get("last_error")
    extracted_data = state.get("output_content", [])
    
    if state.get("urls") and state["current_site_index"] >= len(state["urls"]):
        print(">>> ORCHESTRATOR: All websites processed")
        state["done"] = True
        return state

    print(f"\n>>> ORCHESTRATOR ACTIVE | Step Index: {step_index}")

    # -------- Decide Mode --------
    if extracted_data and step_index == 0:
        mode = "PHASE_2_REPLAN"
    elif last_error:
        mode = "ERROR_RECOVERY"
    else:
        mode = "FRESH_PLAN"

    print(f">>> MODE: {mode}")

    # -------- Build Prompt --------
    if mode == "PHASE_2_REPLAN":
        completed_context = "\n".join(
            f"- Step {s['step_number']}: {s['query']} ({s['agent']})"
            for s in plan
        )

        system_prompt = f"""
You are an Orchestrator Agent.

User Request:
{user_input}

Completed Steps:
{completed_context}

Extracted Data:
{extracted_data[-2:]}

Rules:
- Do NOT repeat completed steps
- Decide next actions
- If finished, end with agent="END"
Return ONLY JSON.
"""

        completed_steps = []

    elif mode == "ERROR_RECOVERY":
        completed_steps = plan[:step_index]

        completed_context = "\n".join(
            f"- Step {s['step_number']}: {s['query']}"
            for s in completed_steps
        )

        system_prompt = f"""
You are an Orchestrator Agent handling a failure.

User Request:
{user_input}

Completed Steps:
{completed_context}

Error:
{last_error}

Rules:
- Fix the error first
- Generate remaining steps only
Return ONLY JSON.
"""

    else:  
        completed_steps = []
        system_prompt = f"""
You are an Orchestrator Agent.

User Request:
{user_input}

Rules:
- Break task into ordered steps
- Assign agents: SEARCH | EXECUTION | OUTPUT | END
- Use natural language
- Return ONLY JSON in this format:

{{
  "steps": [
    {{"agent": "SEARCH", "query": "..."}},
    {{"agent": "EXECUTION", "query": "..."}},
    {{"agent": "OUTPUT", "query": "..."}},
    {{"agent": "END", "query": "..."}}
  ]
}}
"""

    llm_result = call_llm_for_plan(system_prompt)
    new_steps = llm_result.get("steps", [])

    
    final_plan = completed_steps + new_steps

    for i, step in enumerate(final_plan):
        step["step_number"] = i + 1

    print(f">>> PLAN GENERATED | Total Steps: {len(final_plan)}")

    return {
        "plan": final_plan,
        "step_index": len(completed_steps),
        "last_error": None,
        "messages": state.get("messages", []) + [
            {"role": "system", "content": "Plan updated by Orchestrator"}
        ]
    }


In [25]:



tavily_tool = TavilySearch(
    max_results=5,
    tavily_api_key=os.getenv("TAVILY_API")
)


def search_agent(query: str) -> List[Dict]:
    """
    SEARCH AGENT
    Finds free websites using Tavily.
    """
    print(">>> SEARCH AGENT: Searching web...")
    return tavily_tool.invoke(query)
    

In [ ]:



tavily_tool = TavilySearch(
    max_results=5,
    tavily_api_key=os.getenv("TAVILY_API")
)



def search_agent(query: str) -> List[Dict]:
    """
    SEARCH AGENT
    Finds free websites using Tavily.
    """
    print(">>> SEARCH AGENT: Searching web...")
    return tavily_tool.invoke(query)
    

state = orchestrator_agent(state)

for step in state["plan"]:
    if step["agent"] == "SEARCH":
        results = search_agent(step["query"])
        print("\n>>> SEARCH RESULTS:")
        for r in results["results"]:
            print("-", r["url"])




>>> ORCHESTRATOR ACTIVE | Step Index: 0
>>> MODE: FRESH_PLAN
>>> PLAN GENERATED | Total Steps: 4
>>> SEARCH AGENT: Searching web...

>>> SEARCH RESULTS:
- https://vincario.com/vin-decoder/
- https://www.npmjs.com/package/@apiverve/vindecoder?activeTab=readme
- https://dev.carscan.com/api/free-vin-decoder-api
- https://auto.dev/vin
- https://www.decodethis.com/


In [1]:
import os
import json
from typing import Dict, List

from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch


In [2]:
state = {
    # ---- CORE ----
    "user_input": "",
    "plan": [],
    "step_index": 0,
    "last_error": None,
    "output_content": [],
    "messages": [],

    # ---- MULTI-WEBSITE ----
    "urls": [],
    "site_names": [],
    "current_site_index": 0,
    "retry_count": 0,
    "max_retries": 2,
    "aggregated_results": [],

    # ---- OPTIONAL ----
    "execution_messages": [],
    "current_model_index": 0,
    "done": False
}


In [3]:
llm = ChatOpenAI(
    model="gpt-oss-120b",
    api_key=os.getenv("SAMBANOVA_API_KEY"),
    base_url="https://api.sambanova.ai/v1",
    temperature=0
)


In [4]:
def call_llm_for_plan(system_prompt: str) -> dict:
    """
    Only place where Orchestrator talks to LLM.
    """
    response = llm.invoke(system_prompt)
    raw_output = response.content.strip()

    if "```" in raw_output:
        raw_output = raw_output.split("```")[1]

    try:
        return json.loads(raw_output)
    except json.JSONDecodeError:
        raise ValueError(f"Invalid JSON from planner:\n{raw_output}")


In [5]:
tavily_tool = TavilySearch(
    max_results=5,
    tavily_api_key=os.getenv("TAVILY_API")
)

def search_agent(query: str) -> Dict:
    print(">>> SEARCH AGENT: Searching web...")
    return tavily_tool.invoke(query)


In [6]:
def orchestrator_agent(state: Dict) -> Dict:
    """
    ORCHESTRATOR AGENT (Brain)

    Controls:
    - Planning
    - Website iteration
    - Retry / skip
    - Aggregation
    """

    user_input = state["user_input"].replace("{", "{{").replace("}", "}}")
    step_index = state.get("step_index", 0)
    last_error = state.get("last_error")

    print(f"\n>>> ORCHESTRATOR ACTIVE | Site Index: {state['current_site_index']}")

    # ---------------------------------------------------
    # 1️⃣ FINISH CONDITION
    # ---------------------------------------------------
    if state.get("urls") and state["current_site_index"] >= len(state["urls"]):
        print(">>> ORCHESTRATOR: All websites processed")
        state["done"] = True
        return state

    # ---------------------------------------------------
    # 2️⃣ RECEIVE DATA FROM OUTPUT AGENT
    # ---------------------------------------------------
    if state.get("output_content"):
        print(">>> ORCHESTRATOR: Aggregating output")

        state["aggregated_results"].extend(state["output_content"])
        state["output_content"] = []

        state["retry_count"] = 0
        state["current_site_index"] += 1
        state["last_error"] = None

        return state

    # ---------------------------------------------------
    # 3️⃣ ERROR → RETRY OR SKIP
    # ---------------------------------------------------
    if last_error:
        state["retry_count"] += 1

        if state["retry_count"] < state["max_retries"]:
            print(">>> ORCHESTRATOR: Retrying current website")
            return state
        else:
            print(">>> ORCHESTRATOR: Skipping website after retries")
            state["retry_count"] = 0
            state["last_error"] = None
            state["current_site_index"] += 1
            return state

    # ---------------------------------------------------
    # 4️⃣ DISCOVER WEBSITES (RUN ONCE)
    # ---------------------------------------------------
    if not state.get("urls"):
        print(">>> ORCHESTRATOR: Discovering websites")

        search_results = search_agent(
            "Free VIN decoder website using VIN number"
        )

        urls = []
        site_names = []
        
        for r in search_results["results"]:
            urls.append(r["url"])
            site_names.append(r.get("title", r["url"]))
            
        print(urls)
            

        state["urls"] = urls
        state["site_names"] = site_names
        state["current_site_index"] = 0
        state["retry_count"] = 0

        print(f">>> ORCHESTRATOR: Found {len(urls)} websites")
        return state

    # ---------------------------------------------------
    # 5️⃣ GENERATE EXECUTION PLAN (PER SITE)
    # ---------------------------------------------------
    current_url = state["urls"][state["current_site_index"]]

    system_prompt = f"""
You are an Orchestrator Agent.

User Request:
{user_input}

Target Website:
{current_url}

Rules:
- Generate steps ONLY for this website
- Assign agents: EXECUTION → OUTPUT → END
- Do NOT include SEARCH
- Return ONLY JSON

Format:
{{
  "steps": [
    {{"agent": "EXECUTION", "query": "Open {current_url} and perform VIN decoding"}},
    {{"agent": "OUTPUT", "query": "Extract VIN decoding results"}},
    {{"agent": "END", "query": "Finish this website"}}
  ]
}}
"""

    llm_result = call_llm_for_plan(system_prompt)
    steps = llm_result.get("steps", [])

    for i, step in enumerate(steps):
        step["step_number"] = i + 1

    state["plan"] = steps
    state["step_index"] = 0

    print(f">>> ORCHESTRATOR: Plan created for site {current_url}")

    return state


In [7]:
state = orchestrator_agent(state)


>>> ORCHESTRATOR ACTIVE | Site Index: 0
>>> ORCHESTRATOR: Discovering websites
>>> SEARCH AGENT: Searching web...
['https://www.jdpower.com/cars/vin-lookup-and-decoder', 'https://www.decodethevin.com/', 'https://www.decodethis.com/', 'https://driving-tests.org/vin-decoder/', 'https://www.autodna.com/']
>>> ORCHESTRATOR: Found 5 websites


In [9]:
from typing import Dict
from playwright.sync_api import sync_playwright

In [13]:
from playwright.sync_api import sync_playwright, Page, BrowserContext
from typing import Optional
import os


class BrowserManager:
    """
    Singleton Browser Manager

    Responsibilities:
    - Launch headed Playwright browser
    - Maintain ONE active page globally
    - Switch browser context per website
    - Persist session per site using profiles
    """

    _instance = None
    _playwright = None
    _browser: Optional[BrowserContext] = None
    _page: Optional[Page] = None
    _current_site_name: Optional[str] = None
    _headless_mode: bool = False   # Default: HEADED

    # --------------------------------------------------
    # Singleton
    # --------------------------------------------------
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(BrowserManager, cls).__new__(cls)
        return cls._instance

    # --------------------------------------------------
    # Configuration
    # --------------------------------------------------
    def set_headless_mode(self, mode: bool):
        """
        Sets headless mode for NEXT browser launch.
        False = visible browser (recommended)
        """
        self._headless_mode = mode
        print(f">>> Browser headless mode set to: {self._headless_mode}")

    # --------------------------------------------------
    # Browser Lifecycle
    # --------------------------------------------------
    def start_browser(self, url: str, site_name: str) -> str:
        """
        Smart browser starter:
        - Reuses browser if same site
        - Switches profile if site differs
        - Always navigates safely
        """
        try:
            safe_site_name = "".join(
                c for c in site_name if c.isalnum() or c in ("-", "_")
            ).strip() or "default"


            # Reuse existing page
            if self._page and not self._page.is_closed():
                print(f">>> Navigating existing session → {url}")
                try:
                    self._page.goto(url, wait_until="domcontentloaded", timeout=60000)
                    return f"Navigated to {url}"
                except Exception as e:
                    print(f">>> Navigation failed: {e}")
                    self.close_browser()

            # Create profile directory
            profile_dir = os.path.abspath(f"./profiles/{safe_site_name}_profile")
            os.makedirs(profile_dir, exist_ok=True)

            # Start Playwright if needed
            if not self._playwright:
                self._playwright = sync_playwright().start()

            print(f">>> Launching browser profile: {safe_site_name}")

            self._browser = self._playwright.chromium.launch_persistent_context(
                user_data_dir=profile_dir,
                headless=self._headless_mode,
                viewport=None,
                args=[
                    "--start-maximized",
                    "--disable-blink-features=AutomationControlled",
                    "--no-sandbox"
                ],
            )

            self._current_site_name = safe_site_name

            # Get first page
            self._page = self._browser.pages[0] if self._browser.pages else self._browser.new_page()

            self._page.goto(url, wait_until="domcontentloaded", timeout=60000)

            return f"Browser started for {safe_site_name}"

        except Exception as e:
            self.close_browser()
            return f"Browser start error: {str(e)}"

    # --------------------------------------------------
    # Accessors
    # --------------------------------------------------
    def get_page(self) -> Optional[Page]:
        """Returns the active Playwright page."""
        return self._page

    def is_browser_open(self) -> bool:
        return self._page is not None and not self._page.is_closed()

    # --------------------------------------------------
    # Cleanup
    # --------------------------------------------------
    def close_browser(self) -> str:
        """Safely closes browser and Playwright."""
        try:
            if self._page:
                try:
                    self._page.close()
                except:
                    pass

            if self._browser:
                try:
                    self._browser.close()
                except:
                    pass

            if self._playwright:
                try:
                    self._playwright.stop()
                except:
                    pass

            self._page = None
            self._browser = None
            self._playwright = None
            self._current_site_name = None

            return "Browser closed successfully"

        except Exception as e:
            return f"Browser close error: {str(e)}"


# --------------------------------------------------
# GLOBAL INSTANCE
# --------------------------------------------------
browser_manager = BrowserManager()


In [15]:
from langchain_core.tools import tool
from browser_manager import browser_manager

# ===============================
# GLOBAL SOM MEMORY
# ===============================
SOM_STATE = []


# =====================================================
# 0️⃣ OPEN WEBSITE (EXECUTION AGENT ENTRY POINT)
# =====================================================
@tool
def open_website(url: str, site_name: str):
    """
    Opens a website in headed Playwright mode using BrowserManager.
    This is ALWAYS the first step of Execution Agent.
    """
    try:
        return browser_manager.start_browser(url, site_name)
    except Exception as e:
        return f"Error opening website: {str(e)}"

In [16]:
@tool
def enable_vision_overlay():
    """
    Draws red Set-of-Marks (SOM) overlays on visible interactive elements
    and stores them in SOM_STATE for later actions.
    """
    global SOM_STATE
    page = browser_manager.get_page()
    if not page:
        return "Error: No browser page open"

    try:
        elements = page.evaluate("""
        () => {
            document.querySelectorAll('.ai-som-overlay').forEach(e => e.remove());
            let id = 1;
            let data = [];

            const elements = document.querySelectorAll(
                'input, button, textarea, select, a, [role="button"], [onclick], [tabindex]:not([tabindex="-1"])'
            );

            elements.forEach(el => {
                const r = el.getBoundingClientRect();
                const s = window.getComputedStyle(el);

                if (r.width > 5 && r.height > 5 &&
                    s.display !== 'none' && s.visibility !== 'hidden') {

                    let text =
                        el.innerText ||
                        el.placeholder ||
                        el.value ||
                        el.getAttribute("aria-label") ||
                        "";

                    text = text.replace(/\\s+/g, ' ').trim();

                    el.setAttribute("data-ai-id", id);

                    const box = document.createElement("div");
                    box.className = "ai-som-overlay";
                    box.style.cssText = `
                        position:absolute;
                        border:2px solid red;
                        left:${r.left + window.scrollX}px;
                        top:${r.top + window.scrollY}px;
                        width:${r.width}px;
                        height:${r.height}px;
                        z-index:2147483647;
                        pointer-events:none;
                    `;

                    const label = document.createElement("span");
                    label.innerText = id;
                    label.style.cssText = `
                        background:red;
                        color:white;
                        font-size:12px;
                        position:absolute;
                        top:-18px;
                        left:0;
                    `;

                    box.appendChild(label);
                    document.body.appendChild(box);

                    data.push({
                        id,
                        tag: el.tagName.toLowerCase(),
                        type: el.getAttribute("type") || "",
                        text: text.slice(0, 80)
                    });

                    id++;
                }
            });

            return data;
        }
        """)

        SOM_STATE = elements
        return f"SOM enabled: {len(elements)} elements indexed"

    except Exception as e:
        return f"SOM overlay error: {str(e)}"


# =====================================================
# 2️⃣ FIND ELEMENT IDS (SOM SEARCH)
# =====================================================
@tool
def find_element_ids(query: str):
    """
    Finds interactive elements by text/type using SOM memory.
    """
    if not SOM_STATE:
        return "Error: Run enable_vision_overlay first"

    query = query.lower()
    matches = []

    for el in SOM_STATE:
        blob = f"{el['tag']} {el['type']} {el['text']}".lower()
        if query in blob:
            matches.append(f"[ID {el['id']}] <{el['tag']}> {el['text']}")

    if not matches:
        return f"No elements found for '{query}'"

    return "\n".join(matches[:15])



In [17]:
@tool
def click_id(element_id: int):
    """
    Clicks an element using its SOM ID.
    """
    page = browser_manager.get_page()
    if not page:
        return "Error: No browser page open"

    try:
        loc = page.locator(f'[data-ai-id="{element_id}"]').first
        loc.scroll_into_view_if_needed()
        loc.click(force=True)
        return f"Clicked element ID {element_id}"
    except Exception as e:
        return f"Click failed on ID {element_id}: {str(e)}"


@tool
def fill_id(element_id: int, text: str):
    """
    Fills an input field using its SOM ID.
    """
    page = browser_manager.get_page()
    if not page:
        return "Error: No browser page open"

    try:
        loc = page.locator(f'[data-ai-id="{element_id}"]').first
        loc.scroll_into_view_if_needed()
        loc.fill(text)
        return f"Filled element ID {element_id}"
    except Exception as e:
        return f"Fill failed on ID {element_id}: {str(e)}"


# =====================================================
# 4️⃣ PAGE INTERACTION HELPERS
# =====================================================
@tool
def scroll_one_screen():
    """Scrolls down one screen."""
    page = browser_manager.get_page()
    if not page:
        return "Error: No browser page open"
    page.mouse.wheel(0, 700)
    return "Scrolled down"


@tool
def press_key(key: str):
    """Presses a keyboard key (Enter, Tab, Escape, etc.)."""
    page = browser_manager.get_page()
    if not page:
        return "Error: No browser page open"
    page.keyboard.press(key)
    return f"Pressed key: {key}"


# =====================================================
# 5️⃣ PAGE TEXT (CAPTCHA / BLOCK DETECTION)
# =====================================================
@tool
def get_page_text():
    """
    Returns visible page text.
    Used ONLY to detect CAPTCHA, blocks, or errors.
    """
    page = browser_manager.get_page()
    if not page:
        return "Error: No browser page open"

    return page.evaluate("document.body.innerText")[:8000]

In [ ]:
from typing import Dict, Any
from langgraph.graph import StateGraph, END
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda

from browser_manager import browser_manager

# SOM + Browser tools
from tools import (
    enable_vision_overlay,
    find_element_ids,
    click_id,
    fill_id,
    press_key,
    get_page_text,
)

# -------------------------------------------------------------------
# EXECUTION STATE (uses shared global state from orchestration agent)
# -------------------------------------------------------------------
def execution_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """
    Responsibilities:
    - Open website (headed)
    - Interact with page (forms / buttons)
    - Retry twice on failure
    - Return errors / captcha requirement to orchestration agent
    """

    websites = state.get("websites", [])
    current_index = state.get("execution_index", 0)

    if current_index >= len(websites):
        return {
            "execution_done": True
        }

    site = websites[current_index]
    site_name = site["site_name"]
    url = site["url"]

    max_retries = 2
    attempt = 0
    last_error = None

    while attempt < max_retries:
        try:
            print(f"\n[ExecutionAgent] Opening {site_name} (Attempt {attempt + 1})")

            # Always headed
            browser_manager.set_headless_mode(False)
            browser_manager.start_browser(url=url, site_name=site_name)

            # Enable SOM overlay
            enable_vision_overlay()

            # ---- Example interaction logic (VIN use-case) ----
            # Orchestration agent decides WHAT to do
            vin = state.get("vin")

            # Find VIN input
            vin_input_ids = find_element_ids("vin")
            if "No elements found" in vin_input_ids:
                raise RuntimeError("VIN input field not found")

            # Naively pick first ID
            vin_id = int(vin_input_ids.split("[ID:")[1].split("]")[0])
            fill_id(vin_id, vin)

            # Try submit
            submit_ids = find_element_ids("decode")
            if "No elements found" not in submit_ids:
                submit_id = int(submit_ids.split("[ID:")[1].split("]")[0])
                click_id(submit_id)
            else:
                press_key("Enter")

            print(f"[ExecutionAgent] Interaction completed on {site_name}")

            return {
                "execution_index": current_index + 1,
                "execution_results": state.get("execution_results", []) + [
                    {
                        "site_name": site_name,
                        "url": url,
                        "status": "success"
                    }
                ]
            }

        except Exception as e:
            attempt += 1
            last_error = str(e)
            print(f"[ExecutionAgent] Error on {site_name}: {last_error}")

    # ---- After retries exhausted ----
    captcha_flag = "captcha" in (last_error or "").lower()

    return {
        "execution_index": current_index + 1,
        "execution_results": state.get("execution_results", []) + [
            {
                "site_name": site_name,
                "url": url,
                "status": "failed",
                "error": last_error,
                "captcha_required": captcha_flag
            }
        ],
        "messages": state.get("messages", []) + [
            AIMessage(
                content=f"Execution failed for {site_name}. "
                        f"{'Captcha detected.' if captcha_flag else 'Site error.'}"
            )
        ]
    }


# -------------------------------------------------------------------
# LANGGRAPH WRAPPER
# -------------------------------------------------------------------
def build_execution_agent():
    graph = StateGraph(dict)

    graph.add_node("execute_site", RunnableLambda(execution_node))

    graph.set_entry_point("execute_site")
    graph.add_edge("execute_site", END)

    return graph.compile()


In [ ]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_openai import ChatOpenAI


# -------------------------
# INPUT TEXT
# -------------------------

text=[""""
**Best‑price snapshot for “iPhone 15 Pro Max 512 GB” (eBay listings)**  

```json
{
  "eBay": [
    {
      "title": "Apple iPhone 15 Pro Max 256/512GB/1TB Unlocked – Very Good – All Colors",
      "price": "$479.99",
      "price_note": "Free shipping",
      "condition": "New (Shop on eBay)",
      "listing_url": "https://www.ebay.com/itm/..."
    },
    {
      "title": "Apple iPhone 15 Pro Max 256/512GB/1TB Unlocked – Very Good – All Colors",
      "price": "$576.86 – $598.50",
      "price_note": "Free international shipping",
      "condition": "Refurbished – Apple",
      "listing_url": "https://www.ebay.com/itm/..."
    },
    {
      "title": "Apple iPhone 15 Pro Max A2849 512GB Unlocked – Very Good Condition",
      "price": "$598.74",
      "price_note": "Was $1,399.00 – Free international shipping",      
      "condition": "Refurbished – Apple",
      "listing_url": "https://www.ebay.com/itm/..."
    },
    {
      "title": "Apple iPhone 15 Pro Max A2849 512GB Unlocked – Excellent Condition",
      "price": "$635.71",
      "price_note": "Was $951.40 – Free international shipping",        
      "condition": "Refurbished – Apple",
      "listing_url": "https://www.ebay.com/itm/..."
    },
    {
      "title": "Apple iPhone 15 Pro Max A2849 512GB Natural – Excellent Condition – Unlocked",
      "price": "$661.04",
      "price_note": "Was $777.69 – Free international shipping",        
      "condition": "Pre‑Owned – Apple",
      "listing_url": "https://www.ebay.com/itm/..."
    },
    {
      "title": "Apple iPhone 15 Pro Max 256/512GB/1TB Titanium (Unlocked) – Open Box",
      "price": "$813.99 – $890.99",
      "price_note": "Free international shipping",
      "condition": "Open Box – Apple",
      "listing_url": "https://www.ebay.com/itm/..."
    }
  ]
}
```

*Notes*

- Prices are taken directly from the current eBay search results page (sorted by “Best Match”).
- “price_note” includes any shipping or “was” information shown alongside the price.
- The `listing_url` placeholders (`https://www.ebay.com/itm/...`) represent the actual product links; they can be retrieved by clicking the corresponding title on the eBay results page.

These entries represent the lowest‑priced, most‑relevant eBay listings for the iPhone 15 Pro Max 512 GB at the moment of extraction."""]


# -------------------------
# DOCUMENT CONVERSION
# -------------------------

documents = [Document(page_content=t) for t in texts]


# -------------------------
# TEXT SPLITTING
# -------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = splitter.split_documents(documents)


# -------------------------
# EMBEDDINGS
# -------------------------

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# -------------------------
# VECTOR STORE
# -------------------------

vector_store = FAISS.from_documents(docs, embeddings)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})


# -------------------------
# PROMPT
# -------------------------

prompt = ChatPromptTemplate.from_template(
"""
You are a system that converts text into structured JSON.

User Request:
{question}

Rules:
- Output MUST be valid JSON
- Do not include explanations

Context:
{context}
"""
)


# -------------------------
# LLM
# -------------------------

llm = ChatOpenAI(
    model="gpt-oss-120b",
    base_url="https://api.sambanova.ai/v1",
    api_key="da1f3b57-de84-4c9f-b27e-bcdbe79072e6",
    temperature=0
)


# -------------------------
# USER REQUEST
# -------------------------

user_request = "Provide the best prices for iPhone 15 Pro Max 512GB in JSON format with website and price."


# -------------------------
# RETRIEVE CONTEXT
# -------------------------

retrieved_docs = retriever.invoke(user_request)

context = "\n".join([doc.page_content for doc in retrieved_docs])


# -------------------------
# RUN MODEL
# -------------------------

messages = prompt.invoke({
    "context": context,
    "question": user_request
})

response = llm.invoke(messages)

import re

def extract_json(text):
    match = re.search(r'\{.*\}|\[.*\]', text, re.DOTALL)
    if match:
        return json.loads(match.group())
    else:
        raise ValueError("No JSON found")

json_output = extract_json(response.content)

print(json.dumps(json_output, indent=2))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7092.18it/s]


[
  {
    "website": "eBay",
    "price": "$635.71",
    "url": "https://www.ebay.com/itm/..."
  }
]
